# Lab 9: Context Engineering

In this lab you engineer the context of a small language model so that it solves grade-school math problems (GSM8K). No model is trained: the only thing you change is the text the model reads.

- **Part 1: Prompt engineering** (Steps 0–4, ~40 min): instructions, "hocus pocus", reasoning and few-shot examples.
- **Part 2: Context engineering** (Steps 5–7, ~40 min): selecting what goes into the context, irrelevant context, and compressing the context.
- **Wrap-up** (~10 min), and a **bonus** on tool use if you have time left.

Every step follows the same pattern: write a `prompt_builder_*` function that turns a question into a prompt, then measure accuracy and prompt length with `evaluate_model`. The functions you have to write raise `NotImplementedError`.

⏱ An evaluation takes a few minutes. While it runs, read the next step and write its code.

## Setup

We use [`granite-4.0-350m-base`](https://huggingface.co/ibm-granite/granite-4.0-350m-base), a language model of 350 M
parameters by IBM. You can run it in two ways: choose one with `BACKEND` in the next cell, then run the cells below
(only the cell of your backend does something).

- `"transformers"`: Hugging Face `transformers`, with the model as published (16-bit floats). It runs on a GPU if
  there is one. The first run downloads about 700 MB.
- `"llama.cpp"`: `llama-cpp-python`, with the same model quantized to 8 bits (about 380 MB). Faster on a CPU. Install
  it first with `pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu`
  (the extra index has ready-made packages; without it, pip compiles llama.cpp, which needs a C++ compiler). Its
  answers can differ a little from those of the full model.

Both backends define the same two functions, and the rest of the notebook only uses these two: `sample(text, max_len,
stop)` continues `text` (greedily), and `n_tokens(text)` counts its tokens.

In [ ]:
BACKEND = "transformers"  # or "llama.cpp"
STOP = ["Question:"]      # a base model keeps writing: stop as soon as it starts a new question

In [ ]:
if BACKEND == "transformers":
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "ibm-granite/granite-4.0-350m-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto").to(device)  # bf16, as in the checkpoint
    print(f"{model_name}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} M parameters, on {device}")

    def sample(text, max_len=256, stop=STOP):
        # Tokenize prompt
        inputs = tokenizer([text], return_tensors="pt").to(device)

        # Generate answer. Greedy decoding makes results reproducible and comparable between steps
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_len,
            do_sample=False,
            stop_strings=stop or None,
            tokenizer=tokenizer,  # needed by stop_strings
        )

        # Slice off the prompt tokens so you only decode the response
        response = tokenizer.batch_decode(
            outputs[:, inputs.input_ids.shape[1]:],
            skip_special_tokens=True,
        )[0]

        # Cut the response before the first stop string
        for s in stop or []:
            response = response.split(s)[0]
        return response

    def n_tokens(text):
        return len(tokenizer(text).input_ids)

In [ ]:
if BACKEND == "llama.cpp":
    from llama_cpp import Llama

    llm = Llama.from_pretrained(
        repo_id="ibm-granite/granite-4.0-350m-base-GGUF",
        filename="*Q8_0.gguf",    # weights in 8 bits: at 4 bits ("*Q4_K_M.gguf") this model loses most of its arithmetic
        n_ctx=4096,               # the context window: room for the longest few-shot prompts and the answer
        verbose=False,
    )

    def sample(text, max_len=256, stop=STOP):
        # temperature=0 is greedy decoding, as with transformers. The response ends before the first stop string
        return llm(text, max_tokens=max_len, temperature=0.0, stop=stop)["choices"][0]["text"]

    def n_tokens(text):
        return len(llm.tokenize(text.encode()))

This is a _base_ model, which means it hasn't been fine-tuned to follow instructions: it simply continues the text it is
given. Using it like a regular chatbot will give unexpected results.

In [ ]:
print(text := "What's tomorrow's weather ?", sample(text, 16))

In [ ]:
print(text := "Tomorrow's weather is", sample(text, 16))

We use the `GSM8K` dataset. This dataset contains grade-school math problems. Each answer ends with `#### <number>`, which is our ground truth.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")

print("\n".join(f"{k}: {v}" for k, v in dataset["test"][0].items()))

## Part 1: Prompt engineering

**Step 0:** Baseline

We give the raw question to the model and check whether the last number of its response is the right answer. `evaluate_model` also reports the average length of the prompt and of the response, in tokens: context engineering is about accuracy *and* cost.

In [ ]:
import re
import numpy as np
from tqdm import trange

N_SAMPLES = 32  # test questions per evaluation. With a GPU, increase it: fewer questions = noisier results

NUM_RE = re.compile(r"(-?[$0-9.,]{2,})|(-?[0-9]+)")  # return all numbers

def normalize(s: str) -> str:
    s = s.replace(",", "").replace("$", "").strip()
    return s.rstrip(".")

def flexible_extract(text: str) -> str | None:
    matches = NUM_RE.findall(text)
    if not matches:
        return None
    last = [m for m in matches[-1] if m][0]   # take the non-empty group of the LAST match
    return normalize(last)

def is_correct(prediction, ground_truth):
    try:
        return prediction is not None and float(prediction) == float(ground_truth)
    except ValueError:  # e.g. "1.2.3" is not a number
        return False

def evaluate_model(prompt_builder=lambda x: x, n_samples=N_SAMPLES, sampler=sample):
    correct, prompt_len, response_len = 0, 0, 0
    dset = dataset["test"][:n_samples]
    for i in trange(n_samples):
        input_txt = prompt_builder(dset["question"][i])
        resp = sampler(input_txt)
        prompt_len += n_tokens(input_txt)
        response_len += n_tokens(resp)
        ground_truth = normalize(dset["answer"][i].split("####")[-1])
        correct += is_correct(flexible_extract(resp), ground_truth)
    result = {
        "accuracy": correct / n_samples,
        "prompt_tokens": prompt_len / n_samples,
        "response_tokens": response_len / n_samples,
        "n": n_samples,
    }
    print(f"Accuracy: {result['accuracy']:.1%} | prompt: {result['prompt_tokens']:.0f} tokens"
          f" | response: {result['response_tokens']:.0f} tokens")
    return result

In [ ]:
bench = {}  # technique name -> result of evaluate_model

bench["Base"] = evaluate_model()

**Step 1:** System prompt

Let's start by improving the prompt so that it's clear what's expected from the model. A base model has no notion of a system prompt: an instruction is just text placed before the question. For the same reason, we frame the question as `Question: ... Answer:` so that the model knows where its answer should start.

*Optional:* which of the two changes matters more, the instruction or the `Question:`/`Answer:` frame? Remove the instruction and compare.

In [ ]:
def format_question(question):
    return f"Question: {question}\nAnswer:"

SYSTEM_PROMPT = "You are tasked with answering mathematical exercises. Write your answer after the question.\n\n"

def prompt_builder_sys(text):
    return SYSTEM_PROMPT + format_question(text)

print(prompt_builder_sys(dataset["test"][0]["question"]))

In [ ]:
bench["System prompt"] = evaluate_model(prompt_builder_sys)

Let's keep track of the results. `plot_bench` compares all the techniques in `bench`; call it whenever you want.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

def plot_bench(bench):
    names = list(bench)
    acc = np.array([r["accuracy"] for r in bench.values()])
    err = np.array([np.sqrt(r["accuracy"] * (1 - r["accuracy"]) / r["n"]) for r in bench.values()])
    tokens = [r["prompt_tokens"] for r in bench.values()]

    fig, (ax_acc, ax_len) = plt.subplots(1, 2, figsize=(12, 4))
    ax_acc.bar(names, acc * 100, yerr=err * 100, capsize=3, color="#2a78d6")
    ax_acc.set_ylabel("Accuracy (%)")
    ax_len.bar(names, tokens, color="#2a78d6")
    ax_len.set_ylabel("Prompt length (tokens)")
    for ax in (ax_acc, ax_len):
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

plot_bench(bench)

**Step 2:** Hocus Pocus

Let's try further improving the prompt with the "magic" phrases from the lecture. Do they still work on our model?


In [ ]:
def prompt_builder_hocpoc(text):
    system_prompt = "..."
    return system_prompt + format_question(text)

In [ ]:
bench["Hocus pocus"] = evaluate_model(prompt_builder_hocpoc)

**Step 3:** Reasoning

A classic trick is to make the model reason before it answers (zero-shot chain-of-thought, [Kojima et al., 2022](https://arxiv.org/abs/2205.11916)). The intermediate steps the model writes become part of its own context before it produces the final number.

**Task:** write `prompt_builder_cot`

In [ ]:
def prompt_builder_cot(text):
    # YOUR CODE HERE
    raise NotImplementedError()

print(prompt_builder_cot(dataset["test"][0]["question"]))

In [ ]:
bench["Reasoning"] = evaluate_model(prompt_builder_cot)

**Step 4:** Few shots

Instead of describing what we want, we can *show* it: put a few solved examples in the context (in-context learning, [Brown et al., 2020](https://arxiv.org/abs/2005.14165)). The parameters of the model don't change, only its input. We take the examples from the **train** split of GSM8K, so they never overlap with the test questions.


In [ ]:
train = dataset["train"]

def format_example(question, answer):
    # YOUR CODE HERE
    raise NotImplementedError()

def build_fewshot_prompt(question, example_ids, system=SYSTEM_PROMPT, fmt=format_example):
    examples = "".join(fmt(train[int(i)]["question"], train[int(i)]["answer"]) for i in example_ids)
    return system + examples + format_question(question)

def prompt_builder_fewshot(text, k=4):
    return build_fewshot_prompt(text, range(k))  # the first k training examples

print(prompt_builder_fewshot(dataset["test"][0]["question"], k=2))

Now evaluate with a growing number of examples.

⏱ This cell runs 3 evaluations. If you're short on time, keep only `k = 4`.

In [ ]:
K_VALUES = [1, 4, 8]

for k in K_VALUES:
    print(f"{k}-shot")
    bench[f"{k}-shot"] = evaluate_model(lambda text: prompt_builder_fewshot(text, k))

plot_bench(bench)

**Question 4.** How do accuracy and prompt length evolve with `k`? Is the 8-shot prompt worth its cost?

*Your answer here*

## Part 2: Context engineering

Few-shot examples work, but every example costs tokens, and a longer context is not always a better one (*lost in the middle*, *context rot*). Context engineering is about **including what's necessary and removing what isn't**. We look at three ideas from the lecture: selecting the context, the effect of irrelevant context, and compressing the context.

**Step 5:** Selecting the examples

The first `k` training examples are an arbitrary choice. A better idea is to *retrieve* the training examples that are most similar to the current question, just like RAG retrieves documents, or like an agent retrieves relevant memories (StreamBench). We use TF-IDF vectors and cosine similarity: cheap, and good enough to find problems about the same topic.

**Task:**
1. Write `retrieve(question, k)`, which returns the indices of the `k` training questions most similar to `question`, most similar first. *Hint:* the rows of `train_vecs` are L2-normalized, so the dot product of two rows is their cosine similarity.
2. Write `prompt_builder_rag`, which uses the retrieved examples. Where should the most similar example go? Remember *lost in the middle*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
train_vecs = vectorizer.fit_transform(list(train["question"]))  # one row per training question

def retrieve(question, k=4):
    # YOUR CODE HERE
    raise NotImplementedError()

question = dataset["test"][0]["question"]
print("Test question:", question, "\n")
for i in retrieve(question, k=2):
    print("Retrieved:", train[i]["question"], "\n")

In [ ]:
def prompt_builder_rag(text, k=4):
    # YOUR CODE HERE
    raise NotImplementedError()

bench["Retrieved"] = evaluate_model(prompt_builder_rag)

**Question 5.** Compare with `4-shot`. Did the prompt get longer or shorter? Print the retrieved examples for a few test
questions: are they really similar?

*Your answer here*

**Step 6:** Irrelevant context

In real applications, the context often contains information that doesn't matter: an off-topic retrieved document, an old tool output, a long conversation history. How robust is our model to it? Following [Shi et al., 2023](https://arxiv.org/abs/2302.00093), we insert one irrelevant sentence, taken from another problem, into each question.

**Task:** write `add_distractor(question)`, which inserts one sentence from another training problem just before the last sentence of `question` (the actual question). Use `split_sentences`, and draw every random choice from the provided `rng` so that each run uses the same distractors.

In [ ]:
import random
import zlib

def split_sentences(text):
    return re.split(r"(?<=[.!?])\s+", text.strip())

def add_distractor(question):
    rng = random.Random(zlib.crc32(question.encode()))  # same distractor for the same question, run after run
    # YOUR CODE HERE
    raise NotImplementedError()

print(add_distractor(dataset["test"][0]["question"]))

In [ ]:
def prompt_builder_distracted(text, k=4):
    return prompt_builder_fewshot(add_distractor(text), k)

bench["Distractor"] = evaluate_model(prompt_builder_distracted)

*Optional:* Shi et al. found that telling the model it may ignore irrelevant information helps large instruction-tuned
models. Does it help our base model? Write `prompt_builder_distracted_instr`, which adds `Feel free to ignore irrelevant
information given in the questions.` to the system prompt.

In [ ]:
def prompt_builder_distracted_instr(text, k=4):
    # YOUR CODE HERE
    raise NotImplementedError()

bench["Distractor + instr."] = evaluate_model(prompt_builder_distracted_instr)

**Question 6.**
1. Compare `Distractor` with `4-shot`, which uses the same examples without the distractor. What does this suggest about
   stuffing everything we have into the context?
2. (Optional part) Did the instruction help? Shi et al.'s other fix was to use examples that also contain a distractor:
   why could this work better on a base model?

*Your answer here*

**Step 7:** Compressing the context

Our examples contain calculator annotations such as `<<48/2=24>>`, which the authors of GSM8K added to train a model to use a calculator. They repeat what the text already says, so they cost tokens for little benefit. Compression removes such redundancy, but compress too much and you remove what the model needs.

**Task:** write two compressed versions of `format_example`, then evaluate both with the retrieved examples of Step 5:
- `format_example_compact` removes the `<<...>>` annotations.
- `format_example_answer_only` keeps only the final answer (`Answer: The answer is 72.`), without the reasoning.

In [ ]:
def format_example_compact(question, answer):
    # YOUR CODE HERE
    raise NotImplementedError()

def format_example_answer_only(question, answer):
    # YOUR CODE HERE
    raise NotImplementedError()

example = train[0]
for fmt in (format_example, format_example_compact, format_example_answer_only):
    print(f"{fmt.__name__} ({n_tokens(fmt(example['question'], example['answer']))} tokens)")
    print(fmt(example["question"], example["answer"]))

In [ ]:
bench["Compact"] = evaluate_model(
    lambda text: build_fewshot_prompt(text, retrieve(text, 4)[::-1], fmt=format_example_compact)
)
bench["Answer only"] = evaluate_model(
    lambda text: build_fewshot_prompt(text, retrieve(text, 4)[::-1], fmt=format_example_answer_only)
)

plot_bench(bench)

**Question 7.** Which version gives the best accuracy per token? What did the answer-only examples remove that the model
needed? With the tokens saved by compact examples, you could fit more examples in the same budget: try it if you have
time.

*Your answer here*

## Wrap-up

Context engineering is a trade-off between accuracy and context length. Let's put every technique on the same axes.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, r in bench.items():
    ax.scatter(r["prompt_tokens"], r["accuracy"] * 100, s=40, color="#2a78d6")
    ax.annotate(name, (r["prompt_tokens"], r["accuracy"] * 100), xytext=(6, 4), textcoords="offset points", fontsize=9)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())  # 100 and 1000 instead of 10^2 and 10^3
ax.set_xlabel("Prompt length (tokens, log scale)")
ax.set_ylabel("Accuracy (%)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.3)
plt.show()

**Question 8 (discussion).**
1. Which technique gave the largest gain per added token? Which ones would you combine?
2. Which of these techniques would still matter for a large instruction-tuned model, and which would not? (Remember the
   lecture: hocus pocus "does not really work any more".)
3. The third approach of the lecture is multi-agent: several model calls, each with its own short context. How could it
   help our small model on GSM8K?

*Your answer here*

## Bonus: tool use

Small models are bad at arithmetic. Let's give ours a calculator, with the scheme from the lecture: the model writes a command, we stop the generation, run the tool, paste the result into the context, and let the model continue. The GSM8K annotations are calculator calls, so we rewrite them as `<calc>48/2</calc><output>24</output>` to teach the model the syntax through the examples.

In [ ]:
import ast
import operator

OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
             ast.Div: operator.truediv, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calculator(expression):
    """Safely evaluates an arithmetic expression such as '48/2 + 3'. Returns the result as a string."""
    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPERATORS:
            return OPERATORS[type(node.op)](evaluate(node.left), evaluate(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPERATORS:
            return OPERATORS[type(node.op)](evaluate(node.operand))
        raise ValueError(f"unsupported expression: {expression}")
    try:
        expression = expression.replace(",", "").replace("$", "").replace("x", "*").replace("×", "*").strip()
        value = round(evaluate(ast.parse(expression, mode="eval").body), 4)
        return str(int(value)) if value == int(value) else str(value)
    except (SyntaxError, ValueError, TypeError, ZeroDivisionError, OverflowError):
        return "error"

def to_tool_format(answer):
    # "48/2 = <<48/2=24>>24 clips" -> "48/2 = <calc>48/2</calc><output>24</output> clips"
    return re.sub(r"<<([^=<>]*)=([^<>]*)>>(?:\2(?!\.?\d))?", r"<calc>\1</calc><output>\2</output>", answer)

def format_example_tool(question, answer):
    return format_example(question, to_tool_format(answer))

TOOL_PROMPT = SYSTEM_PROMPT.strip() + " You can use a calculator: write <calc>expression</calc> and the result will appear in <output></output>.\n\n"

def prompt_builder_tool(text, k=4):
    return build_fewshot_prompt(text, retrieve(text, k)[::-1], system=TOOL_PROMPT, fmt=format_example_tool)

print(format_example_tool(train[0]["question"], train[0]["answer"]))
print(calculator("48/2 + 3"), calculator("0.2 x 50"), calculator("import os"))

**Task:** complete `sample_with_tools`. Generate until the model closes a `<calc>` tag (use `"</calc>"` as an extra stop string), compute the expression with `calculator`, append `</calc><output>result</output>` to the response and continue, until the model stops for another reason. Keep the total response within `max_len` tokens.

In [ ]:
def sample_with_tools(prompt, max_len=256, max_calls=10):
    response = ""
    # YOUR CODE HERE
    raise NotImplementedError()
    return response

print(sample_with_tools(prompt_builder_tool(dataset["test"][0]["question"])))

In [ ]:
bench["Calculator"] = evaluate_model(prompt_builder_tool, sampler=sample_with_tools)

plot_bench(bench)

**Question B.** Look at a few traces. Does the model call the calculator when it should? Does it copy the result
correctly afterwards? A user doesn't need to see the `<calc>` and `<output>` tags: how would you hide them?

*Your answer here*